# Iterators and Generators in Python

Iterators are one of the core ideas behind Python's looping system. They allow data to be accessed one element at a time and make Python efficient when working with sequences, files, streams, and large datasets. This notebook begins with iterators and then continues into generators.


[Jump to Generators](#generator-section)


## Learning Goals

By the end of this notebook, you should be able to:

- Define what an iterator is.
- Explain the difference between an iterable and an iterator.
- Understand the iterator protocol using `__iter__()` and `__next__()`.
- See how `for` loops use iterators internally.
- Build a custom iterator class.
- Identify iterator exhaustion and common iterator mistakes.
- Understand what a generator is and how `yield` turns a function into a generator.


## 1. Introduction

An iterator is an object that allows sequential access to elements of a collection one item at a time, without exposing the underlying structure.

Python uses iterators internally for many operations such as:

- `for` loops
- Traversing `list`, `tuple`, `set`, and `dict`
- Generators
- File reading
- Many functions in `itertools`


### Why Iterators Matter

Iterators are important because they:

- Enable lazy evaluation, where values are produced only when needed.
- Allow large datasets to be processed without loading everything into memory.
- Provide a standard protocol used across Python.
- Form the foundation for generators and streaming pipelines.
- Make it possible to create custom iterable objects.


## 2. Core Idea (Intuition)

Imagine a playlist of songs.

There are two possible ways to work with it:

| Approach | Description |
|---|---|
| List | Load all songs into memory |
| Iterator | Ask for the next song only when needed |

An iterator behaves like this:

- Next song -> play
- Next song -> play
- Next song -> play
- Stop when no songs remain

This makes iterators both lazy and memory-efficient.


## 3. The Iterator Protocol

Python defines a strict protocol for iterators.

An object is an iterator if it implements these two methods:

1. `__iter__()`
2. `__next__()`

### `__iter__()`

Returns the iterator object itself.

### `__next__()`

Returns the next value.

If no values remain, Python must raise `StopIteration`.

### Visual Model


In [2]:
print("Iterable")
print("   |")
print("   | iter()")
print("   v")
print("Iterator")
print("   |")
print("   | next()")
print("   v")
print("Next Value")


Iterable
   |
   | iter()
   v
Iterator
   |
   | next()
   v
Next Value


## 4. Iterables vs Iterators

| Feature | Iterable | Iterator |
|---|---|---|
| Can be looped | Yes | Yes |
| Has `__iter__()` | Yes | Yes |
| Has `__next__()` | No | Yes |
| Example | `list`, `tuple`, `dict` | result of `iter()` |

Example:


| Concept  | Analogy    | Meaning                                            |
| -------- | ---------- | -------------------------------------------------- |
| Iterable | A book     | A collection you *can start reading from*          |
| Iterator | A bookmark | Keeps track of **where you are currently reading** |


#### What is an Iterator?

An iterator is an object that:

1. Remembers its position

2. Returns the next item

3. Stops when finished

It has two important methods:

`__iter__()`

`__next__()`

In [ ]:
# if somthing is itrable it needs to have spcial method also called dunder method __iter__
# we can check this using inbuilt dir() function
a = [1,2,34,]

print(dir(a))


In [6]:
numbers = [1, 2, 3]

print(numbers)          # iterable
print(iter(numbers))    # iterator


[1, 2, 3]


In [ ]:
#  Inspect iter()
numbers = [1, 2, 3]

it = iter(numbers)

print(type(it))
print(dir(it)) # it has bother __iter__ and __next__ methods that's why its an itrator
print(dir(numbers)) # while and itrable will only have __iter__ method

<class 'list_iterator'>
['__class__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__length_hint__', '__lt__', '__ne__', '__new__', '__next__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__']


In [ ]:
numbers = [10, 20, 30]

it = iter(numbers) # an itrator remembers its state

print(next(it))
print(next(it))
print(next(it))
print(next(it)) # if run out of values it raises stop itration exception
# but when we run for loop on itrable like list it knows how to handle the exception and stops


10
20
30


StopIteration: 

In [ ]:
# Example of for loop itrating over an itrable
a = [1,2,3,4,5]

for i in a:
    print(i)

1
2
3
4
5


In [ ]:
# how for loop works internally
#Python converts for loop  internally into:
a = [1,2,3,4,5]
i_a = iter(a)

while True:
    try:
       i = next(i_a)
       print(i)
    except StopIteration:
        break



1
2
3
4
5


In [ ]:
# each itrator tarcks its own progress
a = [10,20,30]

it1 = iter(a)
it2 = iter(a)

print(next(it1))
print(next(it1))

print(next(it2))

10
20
10


In [ ]:
# An itrator object is coceptually similar to

class ListIterator:

    def __init__(self, data):
        self.data = data
        self.index = 0

    def __next__(self):
        if self.index >= len(self.data):
            raise StopIteration

        value = self.data[self.index]
        self.index += 1
        return value

#     So the iterator remembers position using internal state (usually an index or pointer).

# It is not predicting anything. It just reads the next stored position.

#### What Does `__iter__()` Return?

For an iterator, `__iter__()` returns itself.

Conceptually:
```python
def __iter__(self):
    return self
```


In [21]:
# Example:

it = iter([1,2,3])

print(iter(it) is it)



True


Output:

True

Reason: An iterator is already ready to iterate.

So it returns itself.

In [ ]:
# exmaple to show iterators are forward only they exhust once finished
# you have to create new iterator if you wanna start somthings again



a = [10,20,30]

it = iter(a)

print(next(it))
print(next(it))

print(list(it))

print(next(it))

print(list(it))

10
20
[30]


StopIteration: 

In [ ]:
numbers = [10, 20, 30]

it = iter(numbers) # an itrator remembers its state

print(next(it))
print(next(it))
print(next(it))


10
20
30


In [ ]:
numbers = [10, 20, 30]

it = iter(numbers) # an itrator remembers its state

print(next(it))
print(next(it))


10
20


`numbers` is an iterable. Calling `iter(numbers)` creates an iterator from it.


## 5. Creating an Iterator from an Iterable


In [ ]:
numbers = [10, 20, 30]

iterator = iter(numbers)

print(next(iterator))
print(next(iterator))
print(next(iterator))


Expected output:

```text
10
20
30
```

Explanation:

- `numbers` is an iterable.
- `iter(numbers)` returns an iterator.
- `next(iterator)` returns the next value each time it is called.


## 6. `StopIteration`

When an iterator runs out of values, Python raises `StopIteration`.


In [ ]:
numbers = [1, 2]

it = iter(numbers)

print(next(it))
print(next(it))

try:
    print(next(it))
except StopIteration:
    print("StopIteration")


Expected output:

```text
1
2
StopIteration
```

This exception is Python's way of saying: no more elements are available.


## 7. How a `for` Loop Uses Iterators Internally

A `for` loop is syntactic sugar.


In [ ]:
numbers = [1, 2, 3]

for n in numbers:
    print(n)


Equivalent low-level version:


In [ ]:
numbers = [1, 2, 3]

iterator = iter(numbers)

while True:
    try:
        value = next(iterator)
        print(value)
    except StopIteration:
        break


Key insight: a `for` loop is built from `iter()`, repeated `next()` calls, and automatic `StopIteration` handling.


## 8. Example 1: Iterating a List


In [ ]:
data = ["A", "B", "C"]

it = iter(data)

print(next(it))
print(next(it))
print(next(it))


Expected output:

```text
A
B
C
```


## 9. Example 2: Iterating a String


In [ ]:
text = "AI"

it = iter(text)

print(next(it))
print(next(it))


Expected output:

```text
A
I
```


## 10. Example 3: Iterating a Dictionary


In [ ]:
data = {"a": 1, "b": 2}

it = iter(data)

print(next(it))
print(next(it))


Expected output:

```text
a
b
```

Dictionary iteration returns keys by default.


## 11. Example 4: File Iterator

Files are iterators, which means they can be read one line at a time.


In [ ]:
with open("example.txt", "w", encoding="utf-8") as file:
    file.write("First line\nSecond line\nThird line\n")

with open("example.txt", "r", encoding="utf-8") as file:
    it = iter(file)
    print(next(it))
    print(next(it))


This reads the file lazily, line by line, instead of loading the entire file into memory at once.

In this notebook, the example creates a small file first so the cell runs cleanly.


## 12. Example 5: Using `next()` with a Default

You can provide a default value to prevent `StopIteration` from being raised.


In [ ]:
numbers = [1]

it = iter(numbers)

print(next(it, "END"))
print(next(it, "END"))


Expected output:

```text
1
END
```

If the iterator has no values left, Python returns the default value instead.


## Creating Custom Iterators

A custom iterator class gives you full control over how values are produced.


## 13. Example: Custom Iterator Class


In [ ]:
class Counter:
    def __init__(self, max_value):
        self.max = max_value
        self.current = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.current >= self.max:
            raise StopIteration

        self.current += 1
        return self.current


In [26]:
nums = [1,2,3]

it = iter(nums)

print(next(it))
print(next(it))

nums.append(4)

print(next(it))
print(next(it))

1
2
3
4


Using the iterator:


In [ ]:
counter = Counter(3)

for num in counter:
    print(num)


Expected output:

```text
1
2
3
```


## 14. Memory Advantage of Iterators

Compare these two approaches.


### List


In [ ]:
numbers = [x for x in range(1_000_000)]


This creates one million values in memory immediately.


### Iterator


In [ ]:
numbers = iter(range(1_000_000))


This produces values only when requested, so memory usage stays much smaller.


## 15. Making Custome itrable classes

In [7]:
n = range(1,10)

for i in n:
    print(i) # it does the itration cuse it contains __iter__ method


# print(next(n)) #it's not an itrable but not an iterator cuse it doesn't contain __next__ method of itself


1
2
3
4
5
6
7
8
9


In [ ]:
class Range:

    def __init__(self, start, stop=None, step=1):

        if stop is None:
            start, stop = 0, start

        self.current = start
        self.stop = stop
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):

        if self.current >= self.stop:
            raise StopIteration

        value = self.current
        self.current += self.step
        return value

2
4
6


## Common Mistakes

### Mistake 1: Iterator Exhaustion


In [ ]:
numbers = [1, 2, 3]

it = iter(numbers)

for n in it:
    print(n)

for n in it:
    print(n)




The second loop prints nothing because the iterator has already been consumed.


### Mistake 2: Forgetting `StopIteration`


In [ ]:
class BadIterator:
    def __init__(self):
        self.current = 1

    def __iter__(self):
        return self

    def __next__(self):
        return self.current


This causes infinite iteration because `__next__()` never raises `StopIteration`.


## Mental Model to Remember

- Iterable = container
- Iterator = cursor moving through the container
- `next()` = move the cursor forward
- `StopIteration` = no more elements


## 16. Moving from Iterators to Generators

Now that you understand iterators, the next topic is generators. Generators follow the same idea of producing values one at a time, but Python gives us a simpler way to build them.


<a id="generator-section"></a>

## 17. What is a Generator?

A generator is a special type of function that produces values one at a time instead of returning them all at once.

Normal functions:

- run completely
- return one value
- then terminate

Generators:

- pause execution
- remember their state
- resume later

The keyword that makes this possible is `yield`.

When Python sees `yield`, it transforms the function into a generator function.


## 18. Why Generators Matter in Real Python Development

Generators solve several real problems in Python development.

### 1. Memory Efficiency

Instead of storing millions of values in memory, generators produce values only when needed.

Example:

`range(10_000_000)`

does not store 10 million numbers all at once.

### 2. Streaming Data Processing

Generators are useful when data arrives gradually.

Examples:

- reading large files
- web scraping
- API streaming
- log processing

### 3. Building Pipelines

Generators allow data pipelines where data flows through multiple steps.

Example:

`data -> filter -> transform -> aggregate`

Each stage can be a generator.


## 19. Intuition (Mental Model)

Think of a generator like a movie pause button.

Normal function:

`Start -> Run -> Return -> End`

Generator:

`Start -> yield -> pause`
`resume -> yield -> pause`
`resume -> finish`

Each `yield`:

- sends a value
- freezes the function
- saves all variables and position


## 20. How Python Implements Generators

When Python encounters:

```python
def my_generator():
    yield 1
```

Python creates a generator object.

Internally, it behaves like an iterator with:

- `__iter__()`
- `__next__()`

Calling:

`next(generator)`

continues execution until the next `yield`.

## 21. Generator Syntax

### Generator Function

```python
def generator_function():
    yield value
```

### Creating the Generator

```python
g = generator_function()
```

Important:

The function does not execute yet.

### Consuming Values

```python
next(g)
```

or

```python
for x in g:
    print(x)
```


## 22. Example 1 - Simplest Generator


In [ ]:
def numbers():
    yield 1
    yield 2
    yield 3

g = numbers()

print(next(g))
print(next(g))
print(next(g))


### Execution Trace

First `next()`:

- `yield 1`
- pause

Second `next()`:

- resume
- `yield 2`
- pause

Third `next()`:

- resume
- `yield 3`
- pause

Fourth `next()`:

- `StopIteration`


## 23. Example 2 - Generator with Loop


In [ ]:
def count_up_to(n):
    i = 1
    while i <= n:
        yield i
        i += 1

g = count_up_to(5)

for num in g:
    print(num)


Output:

```text
1
2
3
4
5
```

### Why This Is Powerful

The generator does not create the full list `[1, 2, 3, 4, 5]`.

It generates numbers one by one.


## 24. Example 3 - Square Generator


In [ ]:
def squares(n):
    for i in range(n):
        yield i * i

for s in squares(5):
    print(s)


Output:

```text
0
1
4
9
16
```

### Internal Flow

Iteration steps:

- `i = 0` -> `yield 0`
- `i = 1` -> `yield 1`
- `i = 2` -> `yield 4`
- `i = 3` -> `yield 9`
- `i = 4` -> `yield 16`


## 25. Example 4 - Infinite Generator

Generators can produce infinite sequences.


In [ ]:
def infinite_counter():
    num = 0
    while True:
        yield num
        num += 1

g = infinite_counter()

for _ in range(5):
    print(next(g))


Output:

```text
0
1
2
3
4
```

The generator could continue forever.


## 26. Example 5 - Fibonacci Generator

This is a classic generator example.


In [ ]:
def fibonacci():
    a = 0
    b = 1

    while True:
        yield a
        a, b = b, a + b

fib = fibonacci()

for _ in range(10):
    print(next(fib))


Output:

```text
0
1
1
2
3
5
8
13
21
34
```


## 27. Generator Expressions

Generators also have a short syntax.

### List Comprehension

```python
[x * x for x in range(5)]
```

Result:

```text
[0, 1, 4, 9, 16]
```

### Generator Expression

```python
(x * x for x in range(5))
```

Result:

```text
<generator object>
```

Use it like this:


In [ ]:
gen = (x * x for x in range(5))

for value in gen:
    print(value)


## 28. Memory Difference

### List

```python
nums = [x * x for x in range(10_000_000)]
```

This stores 10 million values in memory.

### Generator

```python
nums = (x * x for x in range(10_000_000))
```

This stores only one value at a time.


## 29. Common Beginner Mistakes

### Mistake 1 - Using `return` Instead of `yield`

Wrong:


In [ ]:
def numbers():
    for i in range(5):
        return i


Output:

```text
0
```

This happens because `return` stops the function completely.

Correct:


In [ ]:
def numbers():
    for i in range(5):
        yield i


### Mistake 2 - Trying to Reuse a Generator


In [ ]:
gen = (x for x in range(3))

for i in gen:
    print(i)

for i in gen:
    print(i)


The second loop prints nothing.

Generators are exhausted after one pass.


## 30. Real Developer Use Cases

### 1. Reading Large Files

```python
def read_lines(file):
    with open(file) as f:
        for line in f:
            yield line
```

### 2. Data Processing Pipelines

```python
def filter_even(nums):
    for n in nums:
        if n % 2 == 0:
            yield n
```

### 3. Streaming APIs

Generators help process:

- logs
- sensor data
- real-time feeds


## 31. Mental Models (Important)

### Model 1 - Pausable Function

`yield = pause + send value`

### Model 2 - Lazy Machine

Generators are machines that produce the next value only when asked.

### Model 3 - Iterators with Less Code

Instead of writing:

- `__iter__()`
- `__next__()`

generators do it automatically.


## 32. Quick Comparison

| Feature | Normal Function | Generator |
|---|---|---|
| Returns | once | multiple times |
| Keyword | `return` | `yield` |
| Memory | may store everything | produces values lazily |
| Execution | runs fully | pauses and resumes |


## 33. Advanced Generator Concepts in Python

Now that the core ideas are clear, the next step is learning some advanced generator features. One of the most useful is `yield from`.

### 1. `yield from`

#### Problem It Solves

Sometimes a generator needs to delegate work to another generator.

Without `yield from`, you must manually loop over the inner generator.

Example without `yield from`:


In [ ]:
def subgen():
    yield 1
    yield 2
    yield 3


def main_gen():
    for value in subgen():
        yield value


This works, but it adds unnecessary boilerplate.

#### Using `yield from`

Python provides:

`yield from <iterable>`

Example:


In [ ]:
def subgen():
    yield 1
    yield 2
    yield 3


def main_gen():
    yield from subgen()


Usage:


In [ ]:
for v in main_gen():
    print(v)


Output:

```text
1
2
3
```

#### What `yield from` Actually Does

Internally, this:

```python
yield from subgen()
```

behaves like:

```python
for value in subgen():
    yield value
```

But it also forwards:

- values
- exceptions
- `send()` calls

So it is much more powerful than a simple loop.


#### Example - Flatten Nested Lists


In [ ]:
def flatten(list_of_lists):
    for sublist in list_of_lists:
        yield from sublist


data = [[1, 2], [3, 4], [5, 6]]

for x in flatten(data):
    print(x)


Output:

```text
1
2
3
4
5
6
```


### 2. Generator Pipelines

Generator pipelines allow multiple generators to process data sequentially.

Think of a factory assembly line.

`data -> filter -> transform -> aggregate`

Each stage is a generator.

#### Example Pipeline

Dataset:

`numbers = range(10)`

Stage 1 - Filter


In [ ]:
def filter_even(nums):
    for n in nums:
        if n % 2 == 0:
            yield n


Stage 2 - Transform


In [ ]:
def square(nums):
    for n in nums:
        yield n * n


Pipeline:


In [ ]:
data = range(10)

evens = filter_even(data)
squares = square(evens)

for value in squares:
    print(value)


Output:

```text
0
4
16
36
64
```

#### Why Pipelines Matter

Advantages:

- memory efficient
- modular
- reusable components
- ideal for big data processing

This pattern is widely used in:

- log processing
- data science pipelines
- ETL systems


### 3. `send()` and Coroutines

Generators normally receive nothing, but Python also allows sending data into a generator.

Method:

`generator.send(value)`

This sends a value to the current `yield` expression.

#### Basic Example


In [ ]:
def echo():
    value = yield
    print("Received:", value)

g = echo()

next(g)          # start generator
g.send("Hello")


Output:

```text
Received: Hello
```

#### Example - Running Accumulator


In [ ]:
def accumulator():
    total = 0
    while True:
        value = yield total
        total += value


acc = accumulator()

print(next(acc))
print(acc.send(10))
print(acc.send(5))
print(acc.send(20))


Output:

```text
0
10
15
35
```

#### How This Works

Flow:

- `yield total` -> pause
- `send(value)` -> resume
- value assigned -> continue

Generators become two-way communication channels.

#### Coroutines

A coroutine is a generator designed to:

- receive data
- process it
- maintain internal state

Example use cases:

- async systems
- pipelines
- event processors


### 4. `itertools` (Powerful Iterator Tools)

Python provides a module specifically for iterator operations.


In [ ]:
import itertools


These are high-performance iterator building blocks.

#### 1. `count()`

Creates an infinite counter.


In [ ]:
counter = itertools.count(10)

for i in counter:
    print(i)
    if i > 15:
        break


Output:

```text
10
11
12
13
14
15
16
```

#### 2. `cycle()`

Repeats a sequence forever.


In [ ]:
cycler = itertools.cycle(["A", "B", "C"])

for i in range(6):
    print(next(cycler))


Output:

```text
A
B
C
A
B
C
```

#### 3. `repeat()`

Repeats the same value.


In [ ]:
for x in itertools.repeat("hi", 3):
    print(x)


Output:

```text
hi
hi
hi
```

#### 4. `chain()`

Combines iterables.


In [ ]:
for x in itertools.chain([1, 2], [3, 4]):
    print(x)


Output:

```text
1
2
3
4
```

#### 5. `islice()`

Slicing for iterators.


In [ ]:
nums = itertools.count()

for x in itertools.islice(nums, 5):
    print(x)


Output:

```text
0
1
2
3
4
```


### 5. Important Generator Patterns

These are patterns used in real software systems.

#### Pattern 1 - Lazy File Reader


In [ ]:
def read_large_file(path):
    with open(path) as f:
        for line in f:
            yield line.strip()


Benefits:

- memory efficient
- handles huge files

#### Pattern 2 - Streaming Filter


In [ ]:
def error_filter(lines):
    for line in lines:
        if "ERROR" in line:
            yield line


#### Pattern 3 - Transformation Stage


In [ ]:
def parse_logs(lines):
    for line in lines:
        yield line.split()


Combined pipeline:


In [ ]:
lines = read_large_file("log.txt")
errors = error_filter(lines)
parsed = parse_logs(errors)

for record in parsed:
    print(record)


#### Pattern 4 - Infinite Data Stream

Example: sensor simulation.


In [ ]:
import random

def sensor():
    while True:
        yield random.random()


#### Pattern 5 - Window Generator

Useful in machine learning preprocessing.


In [ ]:
def sliding_window(data, size):
    for i in range(len(data) - size + 1):
        yield data[i:i + size]


data = [1, 2, 3, 4, 5]

for w in sliding_window(data, 3):
    print(w)


Output:

```text
[1, 2, 3]
[2, 3, 4]
[3, 4, 5]
```


### 6. Key Mental Models

#### Model 1

Generator = lazy sequence machine

`next()` -> next value

#### Model 2

Generator = stateful paused function

`yield` -> pause execution

#### Model 3

Generator pipelines = data assembly line

`input -> generator -> generator -> output`


### 7. Important Interview Insight

Generators are essentially syntactic sugar for iterators.

Python automatically builds these methods behind the scenes:

- `__iter__()`
- `__next__()`
